# CNN steganalysis — Reviewer 1, comment 3

Yedroudj-Net and SRNet against the AMDT stego images and every baseline.

**Runtime → Change runtime type → GPU** before running anything. The module
raises on import without CUDA rather than silently falling back to CPU, so a
skipped GPU can never be mistaken for a completed experiment.

Prerequisite: `run_steganalysis.sh` has finished locally and its
`cnn_payload.zip` has finished uploading to Drive. This notebook does **not**
re-embed — embedding is CPU work and doing it here would waste GPU hours.

In [ ]:
#@title Mount Drive and install
import sys, subprocess, os
IN_COLAB = "google.colab" in sys.modules
assert IN_COLAB, "this notebook is for Colab; run the local half with run_steganalysis.sh"

from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "hydra-core", "omegaconf", "PyWavelets", "wandb"], check=True)

import torch
assert torch.cuda.is_available(), "switch the runtime to GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
#@title Point at the project and the run
from pathlib import Path

PROJECT = Path("/content/amdt_python")   #@param {type:"string"}
RUN_ID  = ""                             #@param {type:"string"}

assert PROJECT.exists(), (
    "copy the amdt_python folder to /content (or clone the repo). "
    "The package is needed for the models, metrics and splits.")
sys.path.insert(0, str(PROJECT / "src"))

DRIVE = Path("/content/drive/MyDrive/amdt-runs")
runs = sorted(p for p in DRIVE.iterdir() if p.is_dir())
RUN = DRIVE / RUN_ID if RUN_ID else runs[-1]
print("run:", RUN.name)

from amdt.utils.seeding import set_seed
set_seed(0)   # before any CUDA context exists

In [ ]:
#@title Stage the cover/stego pairs to local disk
# Reading 40k PNGs straight off mounted Drive dominates epoch time; unzip once
# to /content and read from there.
import zipfile, shutil, time

LOCAL = Path("/content/data")
if not LOCAL.exists():
    src = RUN / "cnn_payload.zip"
    assert src.exists(), (
        f"{src} not found. Either run_steganalysis.sh has not finished, or "
        "Drive is still uploading. Check the Drive icon in your menu bar.")
    t0 = time.time()
    shutil.copy(src, "/content/payload.zip")
    with zipfile.ZipFile("/content/payload.zip") as z:
        z.extractall(LOCAL)
    print(f"staged in {time.time()-t0:.0f}s")

covers = sorted((LOCAL / "cover").glob("*"))
stegos = sorted((LOCAL / "stego").glob("*"))
print(f"{len(covers)} covers, {len(stegos)} stego images")

In [ ]:
#@title Group the stego images by (method, rate)
# Filenames come from StegoStore._fname: "<method>__<rate>__<seed>__<image>.png"
import numpy as np
from collections import defaultdict
from PIL import Image

def load(p):
    return np.asarray(Image.open(p).convert("L"), dtype=np.uint8)

groups = defaultdict(dict)
for p in stegos:
    method, rate, seed, idx = p.stem.split("__")
    groups[(method, float(rate))][int(idx)] = p

cover_by_idx = {i: p for i, p in enumerate(covers)}
for k in sorted(groups):
    print(f"  {k[0]:<12} {k[1]} bpp   {len(groups[k])} images")

In [ ]:
#@title Train and evaluate
#@markdown Yedroudj-Net first: ~500k parameters, converges on a T4 in a couple
#@markdown of hours. SRNet is ~4.8M and needs a curriculum (train at the
#@markdown highest payload, fine-tune downwards) — enable it only if you have
#@markdown the GPU time.
MODEL   = "yedroudj"  #@param ["yedroudj", "srnet"]
EPOCHS  =  60         #@param {type:"integer"}
METHODS = ["AMDT", "LSB", "EA-LSB", "S-UNIWARD", "WOW", "HILL"]  #@param
RATE    = 0.4         #@param {type:"number"}

import pandas as pd
from torch.utils.data import DataLoader
from amdt.data.dataset import cover_wise_split
from amdt.steganalysis.cnn import (PairedStegoDataset, TrainConfig, _collate,
                                   evaluate_cnn, train_cnn)
from amdt.utils.profiling import profile_model
from amdt.utils.seeding import seeded_rng

CKPT = Path("/content/drive/MyDrive/amdt_ckpt")   # survives a disconnect
CKPT.mkdir(parents=True, exist_ok=True)

rows = []
for method in METHODS:
    g = groups.get((method, RATE))
    if not g:
        print(f"skip {method} @ {RATE} — not in the payload"); continue

    idx = sorted(g)
    cov = [load(cover_by_idx[i]) for i in idx]
    stg = [load(g[i]) for i in idx]

    # Cover-wise: a cover and its stego never straddle the split. Putting the
    # stego of a training cover into test is the classic steganalysis leak and
    # inflates accuracy by tens of points.
    sp = cover_wise_split(len(idx), seeded_rng(0, "cnnsplit"), train=0.5, val=0.25)
    mk = lambda s: PairedStegoDataset([cov[i] for i in s], [stg[i] for i in s])

    cfg = TrainConfig(model=MODEL, epochs=EPOCHS, batch_pairs=16, seed=0,
                      checkpoint_dir=str(CKPT / f"{method}_{RATE}_{MODEL}"))
    print(f"\n=== {method} @ {RATE} bpp ===")
    model, hist, prof = train_cnn(mk(sp["train"]), mk(sp["val"]), cfg)

    # Test set touched once, after all selection is complete.
    te = DataLoader(mk(sp["test"]), batch_size=16, collate_fn=_collate)
    m = evaluate_cnn(model, te, next(model.parameters()).device)
    cost = profile_model(model, (1, 1, 512, 512))
    rows.append({"method": method, "rate_bpp": RATE, "detector": MODEL,
                 **m.as_dict(), **prof, **cost})
    print(f"  P_E={m.p_e:.3f}  acc={m.accuracy:.3f}  AUC={m.auc:.3f}")

df = pd.DataFrame(rows)
df.to_csv(RUN / "results_steganalysis_cnn.csv", index=False)
print("\nwritten to", RUN / "results_steganalysis_cnn.csv")
df[["method","accuracy","precision","recall","f1","auc","p_e","md_at_fa5"]].round(3)

In [ ]:
#@title Table for the manuscript
from amdt.evaluation.tables import detection_table, write_table

tex = detection_table(
    {r["method"]: r.to_dict() for _, r in df.set_index("method").iterrows()},
    detector=MODEL,
    caption=(f"CNN steganalysis ({MODEL}) at {RATE} bpp on BOSSBase, cover-wise "
             r"split, selection on validation $P_E$, test set touched once."),
    label="tab:detection_cnn")
p = write_table(tex, RUN, "tab_detection_cnn")
print(p); print(tex)

## Reading the results

`P_E` is the figure of merit, not accuracy: **0.5 means undetectable**, 0.0
means perfectly detected. For a steganographic method, *lower detector
accuracy is better* — the axis runs the opposite way to most ML tables.

Two things to carry into the manuscript honestly:

* A CNN trained on one embedding method at one payload is a **targeted**
  detector, the strongest realistic attacker. If AMDT's `P_E` here is far
  below 0.5, say so; claiming resistance the numbers contradict is what gets
  a paper rejected in the next round.
* If AMDT and LSB land at similar `P_E`, that is the expected result and worth
  stating plainly: T3/T4 defeat *payload-statistics* attacks, not
  residual-based ones. The security claim was never that they would.